# 🏭 AuraFactory — Ollama on Colab

Host Ollama trên Colab free GPU → expose public URL → bot trên Render gọi vào.

**Lưu ý:**
- Colab free session ~12h, ngắt nếu idle quá lâu
- Chạy cell cuối để keep alive
- Khi session mới → chạy lại từ đầu → update URL trên Render

## Step 1: Install Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## Step 2: Start Ollama server (background)

In [ ]:
import subprocess
import time

# Start ollama serve in background
process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)
print('✅ Ollama server started')

## Step 3: Pull model

Dùng `qwen2.5:7b` — cân bằng giữa chất lượng và tốc độ trên T4 GPU.

In [ ]:
!ollama pull qwen2.5:7b
print('\n✅ Model ready!')

## Step 4: Expose qua Cloudflare Tunnel (free, no signup)

Tạo public URL → copy paste vào Render env vars.

In [ ]:
# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print('✅ cloudflared installed')

In [ ]:
import subprocess
import re
import time

# Start tunnel pointing to ollama (port 11434)
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(5)

# Read stderr to find the URL
import threading
url_found = [None]

def read_output(proc):
    for line in proc.stderr:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            url_found[0] = match.group(0)
            break

t = threading.Thread(target=read_output, args=(tunnel_process,))
t.daemon = True
t.start()
t.join(timeout=15)

if url_found[0]:
    print(f'\n{"="*60}')
    print(f'🚀 OLLAMA PUBLIC URL:')
    print(f'\n   {url_found[0]}')
    print(f'\n{"="*60}')
    print(f'\n👉 Copy URL trên → vào Render → Environment Variables:')
    print(f'   OLLAMA_BASE_URL = {url_found[0]}')
    print(f'   LLM_PROVIDER = ollama')
    print(f'   OLLAMA_MODEL = qwen2.5:7b')
    print(f'\n⚠️  URL thay đổi mỗi lần chạy lại notebook!')
else:
    print('❌ Không tìm thấy URL. Chạy lại cell này.')
    # Fallback: manual check
    !cloudflared tunnel --url http://localhost:11434 2>&1 | grep -o 'https://.*trycloudflare.com' | head -1

## Step 5: Test API

In [ ]:
import requests

# Test local
resp = requests.post('http://localhost:11434/api/generate', json={
    'model': 'qwen2.5:7b',
    'prompt': 'Hello! Say hi in one sentence.',
    'stream': False
})
print(f'✅ Response: {resp.json()["response"][:100]}')

## Step 6: Keep Alive

Chạy cell này để Colab không ngắt session. Để chạy nền.

In [ ]:
import time
from IPython.display import clear_output

print('🟢 Keep-alive running. Đừng đóng tab này.')
print(f'   Ollama URL: {url_found[0] if url_found[0] else "check cell trên"}')
print(f'   Model: qwen2.5:7b')
print()

counter = 0
while True:
    counter += 1
    # Ping ollama to keep model loaded
    try:
        requests.get('http://localhost:11434/api/tags', timeout=5)
    except:
        pass
    print(f'\r⏱️ Running {counter} min...', end='', flush=True)
    time.sleep(60)